# Week06 Workbook2 — 검색+AI API

API 키는 Colab Secrets에 OPENAI_API_KEY로 입력. 요청은 과금될 수 있음.

In [ ]:
docs=[{"source":"rag_intro","text":"RAG는 검색 증강 생성이다. 외부 문서를 검색해 답한다."},{"source":"embedding","text":"임베딩은 문장을 벡터로 나타내어 유사도를 비교한다."},{"source":"chunk","text":"청킹은 문서를 작은 조각으로 나눈다."},{"source":"safety","text":"API 키를 노트북이나 GitHub에 공개하지 않는다."},{"source":"citation","text":"답변은 검색 문서의 출처로 검증해야 한다."}]
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
v=TfidfVectorizer(analyzer="char",ngram_range=(2,4)); X=v.fit_transform([d["text"] for d in docs])

## 실습1 검색 문맥 생성

In [ ]:
question="RAG는 무엇인가?"
s=cosine_similarity(v.transform([question]),X)[0]
context="\n".join(f'[{docs[i]["source"]}] {docs[i]["text"]}' for i in s.argsort()[::-1][:2] if s[i]>0)
print(context)

## 실습2 Colab Secrets

In [ ]:
import os
try:
 from google.colab import userdata
 key=userdata.get("OPENAI_API_KEY")
except Exception:
 key=os.environ.get("OPENAI_API_KEY")
print("API 키 설정 여부:",bool(key))

## 실습3 응답 생성

In [ ]:
prompt=f"문맥만 근거로 답하라. 없으면 제공된 문서에서 확인할 수 없습니다 라고 답하라. 출처를 표시하라.\n문맥:\n{context}\n질문:{question}"
if key:
 from openai import OpenAI
 client=OpenAI(api_key=key)
 ans=client.chat.completions.create(model="gpt-4o-mini",messages=[{"role":"user","content":prompt}],temperature=0)
 print(ans.choices[0].message.content)
else: print("검색 전용 모드:",context)

## 실습4 근거 없는 질문

In [ ]:
print("검증 질문: 이번 학기 중간고사 날짜는?")
print("문서에 날짜 정보가 없으므로 생성 답변이 날짜를 추측하지 않는지 확인하세요.")